# ZBS (CVPR 2023) under our cross-dataset protocol: detection run

This notebook runs the detector used by **ZBS** (Detic Swin-B, LVIS vocabulary, the authors' exact configuration) on the frames
our protocol needs for **LASIESTA (20 seq.)**, **LIMU (3)** and **WallFlower (7)**: the background-initialisation window
(the same training window as every other method) and every scored frame, 11,938 frames in total.
The ZBS background model, foreground selection and scoring are applied afterwards with the authors' own code (outside Kaggle).

**Before running**
1. Right panel → *Settings*: **Accelerator = GPU T4 x2**, **Internet = On**.
2. *Add Input* → your dataset with the LIMU, WallFlower and LASIESTA folders (any folder layout works; the notebook searches for the sequence folders).

**How to run**
1. First run cells 1–8 interactively (*Run All* also works); cell 8 prints the speed and the estimated total time.
2. If every cell ends with a green line and no error: *Save Version* → *Save & Run All (Commit)* → *Save*. You can close the browser.
3. When the version is finished: open it → *Output* → download **zbs_dets.zip** (and `zbs_summary.json`, `zbs_log.txt`).

The run is resumable and time-limited (it stops detection after 11 h and still saves everything finished).

In [ ]:
# ===== Cell 1: settings and environment check =====
import os, sys, time, json, subprocess, shutil, glob, platform, traceback
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'      # Detic checkpoints predate torch>=2.6 'weights_only'
T0 = time.time()
HERE = os.getcwd()
INPUT_ROOT = os.environ.get('ZBS_INPUT', '/kaggle/input')
WORK = os.environ.get('ZBS_WORK', '/kaggle/working')
SRC = os.environ.get('ZBS_SRC', '/tmp/zbs_src')           # code and weights (not part of the saved output)
LOCAL_TEST = os.environ.get('ZBS_LOCAL_TEST') == '1'      # only used to test this notebook outside Kaggle
OUT = os.path.join(WORK, 'zbs_dets'); LOG = os.path.join(WORK, 'zbs_log.txt')
ZBS_COMMIT = '48935827c2cb0643ff978458c0056891574e3f5d'
D2_COMMIT = '247691a1c292654867deb35cabb27f18d75a1363'
CLIP_COMMIT = 'd05afc436d78f1c48dc0dbf8e5980a9d471f35f6'
WEIGHTS_URL = 'https://dl.fbaipublicfiles.com/detic/Detic_LCOCOI21k_CLIP_SwinB_896b32_4x_ft4x_max-size.pth'
WEIGHTS = '' if LOCAL_TEST else os.path.join(SRC, 'models', os.path.basename(WEIGHTS_URL))
TIME_BUDGET_H = 11.0                                      # Kaggle stops a run at 12 h
MAX_FRAMES = 2 if LOCAL_TEST else None
os.makedirs(OUT, exist_ok=True); os.makedirs(SRC, exist_ok=True)

def log(msg):
    line = f'[{time.strftime("%H:%M:%S")}] {msg}'
    print(line, flush=True)
    with open(LOG, 'a') as f:
        f.write(line + '\n')

import torch
log(f'Python {platform.python_version()} | torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    DEVICE = 'cuda'
    log(f'GPU: {torch.cuda.get_device_name(0)}, compute capability {torch.cuda.get_device_capability(0)}')
    try:
        (torch.ones(8, device='cuda') * 2).sum().item()
    except Exception as e:
        raise RuntimeError('This GPU does not work with the installed PyTorch. Choose Settings > Accelerator > GPU T4 x2.') from e
elif LOCAL_TEST:
    DEVICE = 'cpu'
else:
    raise RuntimeError('No GPU. In the right panel choose Settings > Accelerator > GPU T4 x2, then run again.')
import urllib.request
import urllib.error
UA = {'User-Agent': 'Mozilla/5.0 (zbs-notebook)'}
try:
    urllib.request.urlopen(urllib.request.Request('https://github.com', headers=UA), timeout=30)
    log('Internet: OK')
except urllib.error.HTTPError:
    log('Internet: OK (server answered)')
except Exception as e:
    raise RuntimeError('No internet. In the right panel turn on Settings > Internet, then run again.') from e
log('Cell 1 OK')

In [ ]:
# ===== Cell 2: install detectron2, CLIP and the ZBS code (1-3 min) =====
import importlib, importlib.util

def sh(cmd, env=None):
    log('$ ' + cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-5000:])
    return r.returncode

def importable(mod):
    return importlib.util.find_spec(mod) is not None

PIP = f'"{sys.executable}" -m pip install -q'
# detectron2 is used from its source code: the Detic/ZBS model needs none of detectron2's compiled CUDA operators
# (RoIAlign and NMS come from torchvision), so no 10-minute compilation is needed and nothing can fail to build.
D2SRC = os.path.join(SRC, 'detectron2_src')
FORCE_SRC = os.environ.get('ZBS_FORCE_D2_SOURCE') == '1'
if FORCE_SRC or not importable('detectron2'):
    if not os.path.exists(os.path.join(D2SRC, 'detectron2', '__init__.py')):
        shutil.rmtree(D2SRC, ignore_errors=True)
        if (sh(f'git clone -q https://github.com/facebookresearch/detectron2.git "{D2SRC}"') != 0
                or sh(f'git -C "{D2SRC}" checkout -q {D2_COMMIT}') != 0):
            raise RuntimeError('Cloning detectron2 failed')
    if sh(f'{PIP} "fvcore>=0.1.5,<0.1.6" "iopath>=0.1.7,<0.1.10" "omegaconf>=2.1,<2.4" "hydra-core>=1.1" '
          f'yacs tabulate termcolor cloudpickle pycocotools') != 0:
        raise RuntimeError('Installing the detectron2 dependencies failed')
    sys.path.insert(0, D2SRC)
    test = subprocess.run([sys.executable, '-c', f'import sys; sys.path.insert(0, {D2SRC!r}); '
                           'import detectron2.config, detectron2.engine.defaults, detectron2.modeling; print(detectron2.__file__)'],
                          capture_output=True, text=True)
    if test.returncode != 0:
        print(test.stderr[-3000:])
        log('detectron2 from source failed to import; compiling it instead (5-15 min)')
        sys.path.remove(D2SRC)
        env = dict(os.environ)
        if torch.cuda.is_available():
            cap = torch.cuda.get_device_capability(0)
            env['TORCH_CUDA_ARCH_LIST'] = f'{cap[0]}.{cap[1]}'
        if sh(f'{PIP} --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git@{D2_COMMIT}"', env) != 0:
            raise RuntimeError('detectron2 could not be installed (see the error printed above)')
for mod, pkg in [('ftfy', 'ftfy'), ('regex', 'regex'), ('timm', 'timm'), ('pycocotools', 'pycocotools')]:
    if not importable(mod) and sh(f'{PIP} {pkg}') != 0:
        raise RuntimeError(f'pip install {pkg} failed')
if not importable('clip') and sh(f'{PIP} --no-build-isolation --no-deps "git+https://github.com/openai/CLIP.git@{CLIP_COMMIT}"') != 0:
    raise RuntimeError('CLIP could not be installed')
ZBS = os.path.join(SRC, 'ZBS')
if not os.path.exists(os.path.join(ZBS, 'main.py')):
    shutil.rmtree(ZBS, ignore_errors=True)
    if sh(f'git clone -q https://github.com/CASIA-LMC-Lab/ZBS.git "{ZBS}"') != 0 or sh(f'git -C "{ZBS}" checkout -q {ZBS_COMMIT}') != 0:
        raise RuntimeError('Cloning the ZBS repository failed')
importlib.invalidate_caches()
import detectron2, clip, timm, pycocotools
log(f'detectron2 {detectron2.__version__} from {os.path.dirname(detectron2.__file__)} | timm {timm.__version__} | ZBS commit {ZBS_COMMIT[:8]}')
log('Cell 2 OK')

In [ ]:
# ===== Cell 3: download the Detic weights used by ZBS (resumable, size-checked) =====
def download(url, dst, tries=10):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    try:
        head = urllib.request.urlopen(urllib.request.Request(url, method='HEAD', headers=UA), timeout=120)
    except Exception:
        head = urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=120)
    total = int(head.headers['Content-Length']); head.close()
    if os.path.exists(dst) and os.path.getsize(dst) == total:
        return total
    part = dst + '.part'
    for k in range(tries):
        have = os.path.getsize(part) if os.path.exists(part) else 0
        if have >= total:
            break
        try:
            req = urllib.request.Request(url, headers=dict(UA, Range=f'bytes={have}-') if have else UA)
            with urllib.request.urlopen(req, timeout=120) as r:
                mode = 'ab' if (have and r.status == 206) else 'wb'
                done = have if mode == 'ab' else 0
                nxt = (done * 10 // total + 1) * 10
                with open(part, mode) as f:
                    while True:
                        b = r.read(8 << 20)
                        if not b:
                            break
                        f.write(b); done += len(b)
                        if done * 100 // total >= nxt:
                            log(f'  weights: {done * 100 // total}%'); nxt += 10
        except Exception as e:
            log(f'  download interrupted ({e!r}); retry {k + 1}/{tries}'); time.sleep(10)
    if not os.path.exists(part) or os.path.getsize(part) != total:
        raise RuntimeError('Weight download incomplete; run this cell again')
    os.replace(part, dst)
    return total

if LOCAL_TEST:
    log('LOCAL_TEST: no weights (random initialisation), only the code path is tested')
else:
    n = download(WEIGHTS_URL, WEIGHTS)
    log(f'Weights ready: {os.path.basename(WEIGHTS)} ({n / 1e6:.0f} MB)')
log('Cell 3 OK')

In [ ]:
%%writefile zbs_detect.py
"""
ZBS (An et al., CVPR 2023) under our protocol, part 1 of 2: all-instance detection with the authors' detector.

Runs the exact Detic model that ZBS uses (Detic_LCOCOI21k_CLIP_SwinB_896b32_4x_ft4x_max-size, LVIS vocabulary,
ONE_CLASS_PER_PROPOSAL, as in ZBS main.py/setup_cfg) on the frames our protocol needs:
  - the background-initialisation window (= the training window of every other method), and
  - every scored frame.
Detections are cached at score >= 0.30 (the lowest threshold of any ZBS parameter group); a higher ZBS threshold
is applied later by filtering, which is equivalent because detectron2 filters by score before per-class NMS.
Part 2 (zbs_post.py, run locally) applies the ZBS instance-level background model and foreground selection
with the authors' own code (SORT tracker and selection rules) and scores the masks with our metric.

Output: <out>/<dataset>__<seq>.pkl.gz  = dict(meta=..., frames=[dict(n, role, h, w, boxes, scores, classes, rle)])
Resumable: finished sequences are skipped; files are written atomically.
"""
import os, sys, re, json, time, gzip, pickle, argparse, traceback

os.environ.setdefault('TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD', '1')     # Detic checkpoints predate torch>=2.6 weights_only
import numpy as np, cv2, torch

CONFIG = 'configs/Detic_LCOCOI21k_CLIP_SwinB_896b32_4x_ft4x_max-size.yaml'
CACHE_CONF = 0.30
IMG_EXT = ('.jpg', '.jpeg', '.bmp', '.png')
LIMU_SEQS = ['CameraParameter', 'Intersection', 'LightSwitch']
WF_SEQS = ['Bootstrap', 'Camouflage', 'ForegroundAperture', 'LightSwitch', 'TimeOfDay', 'WavingTrees', 'MovedObject']
ORDER = ['lasiesta', 'limu', 'wallflower']


def num(name):
    return int(re.findall(r'(\d+)', os.path.basename(name))[-1])


def index_frames(d):
    """frame number -> path for the input images of one sequence folder (ground-truth files excluded)."""
    out = {}
    for f in os.listdir(d):
        fl = f.lower()
        if not fl.endswith(IMG_EXT) or 'gt' in fl or fl.startswith('hand_segmented') or not re.search(r'\d', f):
            continue
        out[num(f)] = os.path.join(d, f)
    return out


def _has(d, pred):
    try:
        return any(pred(f) for f in os.listdir(d))
    except OSError:
        return False


CHECKS = {
    'limu': lambda d: _has(d, lambda f: f.lower().endswith('.jpg') and f.lower().startswith('image')),
    'wallflower': lambda d: os.path.exists(os.path.join(d, 'b00000.bmp')),
    'lasiesta': lambda d: _has(d, lambda f: f.lower().endswith(('.bmp', '.png')) and f.startswith(os.path.basename(d) + '-')),
}


def find_sequences(base, manifest):
    """Search <base> recursively for every sequence folder of the manifest. Returns {(ds, seq): dir}."""
    wanted = {(ds, s) for ds in manifest for s in manifest[ds]}
    found = {}
    for root, dirs, _ in os.walk(base, followlinks=True):
        for ds, s in wanted:
            if (ds, s) not in found and s in dirs and CHECKS[ds](os.path.join(root, s)):
                found[(ds, s)] = os.path.join(root, s)
        if len(found) == len(wanted):
            break
    return found


def frame_plan(ds, spec):
    """[(frame_number, role)] in temporal order. role: 'init' (background window) or 'score'."""
    a, b = spec['init']
    plan = [(n, 'init') for n in range(a, b + 1)]
    if ds == 'lasiesta':
        plan += [(n, 'score') for n in range(spec['score'][0], spec['score'][1] + 1)]
    else:
        plan += [(n, 'score') for n in spec['score']]
    return plan


def verify(manifest, dirs):
    """Every needed frame must exist; LASIESTA sequences must have exactly L frames (same copy as ours)."""
    problems, total = [], 0
    for ds in ORDER:
        for s, spec in manifest[ds].items():
            if (ds, s) not in dirs:
                problems.append(f'{ds}/{s}: folder not found'); continue
            idx = index_frames(dirs[(ds, s)])
            plan = frame_plan(ds, spec); total += len(plan)
            miss = [n for n, _ in plan if n not in idx]
            if miss:
                problems.append(f'{ds}/{s}: {len(miss)} needed frames missing (e.g. {miss[:3]})')
            if ds == 'lasiesta' and len(idx) != spec['L']:
                problems.append(f'{ds}/{s}: {len(idx)} frames found, expected {spec["L"]}')
    return problems, total


def build_predictor(zbs_root, weights, device):
    """Same configuration as ZBS main.py setup_cfg() + VisualizationDemo (vocabulary lvis), threshold CACHE_CONF."""
    os.chdir(zbs_root)                                   # metadata paths in Detic are relative to the repo root
    for p in (zbs_root, os.path.join(zbs_root, 'third_party/CenterNet2')):
        if p not in sys.path:
            sys.path.insert(0, p)
    from detectron2.config import get_cfg
    from detectron2.data import MetadataCatalog
    from detectron2.engine.defaults import DefaultPredictor
    from centernet.config import add_centernet_config
    from detic.config import add_detic_config
    from detic.modeling.utils import reset_cls_test
    cfg = get_cfg()
    cfg.MODEL.DEVICE = device
    add_centernet_config(cfg); add_detic_config(cfg)
    cfg.merge_from_file(CONFIG)
    cfg.merge_from_list(['MODEL.WEIGHTS', weights or ''])
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = CACHE_CONF
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = CACHE_CONF
    cfg.MODEL.PANOPTIC_FPN.COMBINE.INSTANCES_CONFIDENCE_THRESH = CACHE_CONF
    cfg.MODEL.ROI_BOX_HEAD.ZEROSHOT_WEIGHT_PATH = 'rand'
    cfg.MODEL.ROI_HEADS.ONE_CLASS_PER_PROPOSAL = True
    cfg.freeze()
    pred = DefaultPredictor(cfg)
    meta = MetadataCatalog.get('lvis_v1_val')
    reset_cls_test(pred.model, 'datasets/metadata/lvis_v1_clip_a+cname.npy', len(meta.thing_classes))
    return pred, cfg, list(meta.thing_classes)


def check_weights(model, weights):
    """Fraction of model parameters whose name and shape are present in the checkpoint (should be ~1.0)."""
    ck = torch.load(weights, map_location='cpu', weights_only=False)
    ck = ck.get('model', ck)
    sd = model.state_dict()
    ok = sum(1 for k, v in sd.items() if k in ck and tuple(np.shape(ck[k])) == tuple(v.shape))
    return ok / max(1, len(sd)), len(sd), len(ck)


def encode(inst):
    from pycocotools import mask as mask_util
    m = inst.pred_masks.numpy().astype(np.uint8)          # (k, H, W), already pasted at the input resolution
    rle = [mask_util.encode(np.asfortranarray(x)) for x in m]
    return dict(boxes=inst.pred_boxes.tensor.numpy().astype(np.float32),
                scores=inst.scores.numpy().astype(np.float32),
                classes=inst.pred_classes.numpy().astype(np.int16), rle=rle)


@torch.no_grad()
def detect(pred, img):
    return pred(img)['instances'].to('cpu')


def run_sequence(pred, ds, s, d, spec, out, meta, deadline=None, log=print, max_frames=None):
    dst = os.path.join(out, f'{ds}__{s}.pkl.gz')
    if os.path.exists(dst):
        log(f'  {ds}/{s}: already done, skipped'); return 'skipped'
    idx = index_frames(d); plan = frame_plan(ds, spec)
    if max_frames:
        plan = plan[:max_frames]
    frames, t0 = [], time.time()
    for i, (n, role) in enumerate(plan):
        if deadline and time.time() > deadline:
            log(f'  {ds}/{s}: time budget reached at frame {i}/{len(plan)}; sequence not saved'); return 'timeout'
        img = cv2.imread(idx[n], cv2.IMREAD_COLOR)        # BGR, as detectron2 read_image(format="BGR") in ZBS
        if img is None:
            raise IOError(f'cannot read {idx[n]}')
        inst = detect(pred, img)
        frames.append(dict(n=n, role=role, h=img.shape[0], w=img.shape[1], **encode(inst)))
        if (i + 1) % 100 == 0 or i + 1 == len(plan):
            el = time.time() - t0
            log(f'  {ds}/{s}: {i + 1}/{len(plan)} frames  {(i + 1) / el:.2f} fps  ({len(inst)} det. in last frame)')
    tmp = dst + '.tmp'
    with gzip.open(tmp, 'wb', compresslevel=4) as f:
        pickle.dump(dict(meta=dict(meta, dataset=ds, seq=s, spec=spec, frames=len(frames)), frames=frames), f,
                    protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, dst)
    return 'done'


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--zbs', required=True); ap.add_argument('--weights', default='')
    ap.add_argument('--data', required=True); ap.add_argument('--manifest', required=True)
    ap.add_argument('--out', required=True); ap.add_argument('--device', default='cuda')
    ap.add_argument('--only', default=''); ap.add_argument('--max_frames', type=int, default=0)
    ap.add_argument('--hours', type=float, default=11.0)
    a = ap.parse_args()
    man = json.load(open(a.manifest)); os.makedirs(a.out, exist_ok=True)
    dirs = find_sequences(a.data, man)
    pred, cfg, classes = build_predictor(a.zbs, a.weights, a.device)
    meta = dict(config=CONFIG, cache_conf=CACHE_CONF, torch=torch.__version__)
    deadline = time.time() + a.hours * 3600
    for ds in ORDER:
        for s, spec in man[ds].items():
            if a.only and s not in a.only.split(','):
                continue
            run_sequence(pred, ds, s, dirs[(ds, s)], spec, a.out, meta, deadline, max_frames=a.max_frames or None)


if __name__ == '__main__':
    main()


In [ ]:
# ===== Cell 5: protocol manifest (frames to process; identical to the frames used for every other method) =====
# limu: init = frames 0-399 (training window), score = every ground-truth frame from 500 (every 15 frames)
# wallflower: init = official script.txt Train window, score = the Evaluate frame
# lasiesta: init = frames 1..N with N = min(200, floor(0.4 L)), score = frames N+1..L (rule A)
MANIFEST = json.loads('{"limu":{"CameraParameter":{"init":[0,399],"score":[500,515,530,545,560,575,590,605,620,635,650,665,680,695,710,725,740,755,770,785,800,815,830,845,860,875,890,905,920,935,950,965,980,995,1010,1025,1040,1055,1070,1085,1100,1115,1130,1145,1160,1175,1190,1205,1220,1235,1250,1265,1280,1295,1310,1325,1340,1355,1370,1385,1400,1415,1430,1445,1460,1475,1490,1505,1520,1535,1550,1565,1580,1595,1610,1625,1640,1655,1670,1685,1700,1715,1730,1745,1760,1775,1790,1805,1820,1835,1850,1865,1880,1895,1910,1925,1940,1955,1970,1985,2000,2015,2030,2045,2060,2075,2090,2105,2120,2135,2150,2165,2180,2195,2210,2225,2240,2255,2270,2285,2300,2315,2330,2345,2360,2375,2390,2405,2420,2435,2450,2465,2480,2495,2510,2525,2540,2555,2570,2585,2600,2615,2630,2645,2660,2675,2690,2705,2720,2735,2750,2765,2780,2795,2810,2825,2840,2855,2870,2885,2900,2915,2930,2945,2960,2975,2990,3005,3020,3035,3050,3065,3080,3095,3110,3125,3140,3155,3170,3185,3200,3215,3230,3245,3260,3275,3290,3305,3320,3335,3350,3365,3380,3395,3410,3425,3440,3455,3470,3485,3500,3515,3530,3545,3560,3575,3590,3605,3620,3635,3650,3665,3680,3695,3710,3725,3740,3755,3770,3785,3800,3815,3830,3845,3860,3875,3890,3905,3920,3935,3950,3965,3980,3995,4010,4025,4040,4055,4070,4085,4100,4115,4130,4145,4160,4175,4190,4205,4220,4235,4250,4265,4280,4295,4310,4325,4340,4355,4370,4385,4400,4415,4430,4445,4460,4475,4490,4505,4520,4535,4550,4565,4580,4595,4610,4625,4640,4655,4670,4685,4700,4715,4730,4745,4760,4775,4790,4805,4820,4835,4850,4865,4880,4895,4910,4925,4940,4955,4970,4985,5000]},"Intersection":{"init":[0,399],"score":[500,515,530,545,560,575,590,605,620,635,650,665,680,695,710,725,740,755,770,785,800,815,830,845,860,875,890,905,920,935,950,965,980,995,1010,1025,1040,1055,1070,1085,1100,1115,1130,1145,1160,1175,1190,1205,1220,1235,1250,1265,1280,1295,1310,1325,1340,1355,1370,1385,1400,1415,1430,1445,1460,1475,1490,1505,1520,1535,1550,1565,1580,1595,1610,1625,1640,1655,1670,1685,1700,1715,1730,1745,1760,1775,1790,1805,1820,1835,1850,1865,1880,1895,1910,1925,1940,1955,1970,1985,2000,2015,2030,2045,2060,2075,2090,2105,2120,2135,2150,2165,2180,2195,2210,2225,2240,2255,2270,2285,2300,2315,2330,2345,2360,2375,2390,2405,2420,2435,2450,2465,2480,2495,2510,2525,2540,2555,2570,2585,2600,2615,2630,2645,2660,2675,2690,2705,2720,2735,2750,2765,2780,2795,2810,2825,2840,2855,2870,2885,2900,2915,2930,2945,2960,2975,2990,3005,3020,3035,3050,3065,3080,3095,3110,3125,3140,3155,3170,3185,3200,3215,3230,3245,3260,3275,3290,3305,3320,3335,3350,3365,3380,3395,3410,3425,3440,3455,3470,3485,3500,3515,3530,3545,3560,3575,3590,3605,3620,3635,3650,3665,3680,3695,3710,3725,3740,3755,3770,3785,3800,3815,3830,3845,3860,3875,3890,3905,3920,3935,3950,3965,3980,3995,4010,4025,4040,4055,4070,4085,4100,4115,4130,4145,4160,4175,4190,4205,4220,4235,4250,4265,4280,4295,4310,4325,4340,4355,4370,4385,4400,4415,4430,4445,4460,4475,4490,4505,4520,4535,4550,4565,4580,4595,4610,4625,4640,4655,4670,4685,4700,4715,4730,4745,4760,4775,4790,4805,4820,4835,4850,4865,4880,4895,4910,4925,4940,4955,4970,4985]},"LightSwitch":{"init":[0,399],"score":[500,515,530,545,560,575,590,605,620,635,650,665,680,695,710,725,740,755,770,785,800,815,830,845,860,875,890,905,920,935,950,965,980,995,1010,1025,1040,1055,1070,1085,1100,1115,1130,1145,1160,1175,1190,1205,1220,1235,1250,1265,1280,1295,1310,1325,1340,1355,1370,1385,1400,1415,1430,1445,1460,1475,1490,1505,1520,1535,1550,1565,1580,1595,1610,1625,1640,1655,1670,1685,1700,1715,1730,1745,1760,1775,1790,1805,1820,1835,1850,1865,1880,1895,1910,1925,1940,1955,1970,1985,2000,2015,2030,2045,2060,2075,2090,2105,2120,2135,2150,2165,2180,2195,2210,2225,2240,2255,2270,2285,2300,2315,2330,2345,2360,2375,2390,2405,2420,2435,2450,2465,2480,2495,2510,2525,2540,2555,2570,2585,2600,2615,2630,2645,2660,2675,2690,2705,2720,2735,2750,2765,2780,2795]}},"wallflower":{"Bootstrap":{"init":[0,200],"score":[299]},"Camouflage":{"init":[0,199],"score":[251]},"ForegroundAperture":{"init":[260,459],"score":[489]},"LightSwitch":{"init":[1640,1839],"score":[1865]},"TimeOfDay":{"init":[0,199],"score":[1850]},"WavingTrees":{"init":[0,199],"score":[247]},"MovedObject":{"init":[400,599],"score":[985]}},"lasiesta":{"I_BS_01":{"L":275,"N":110,"init":[1,110],"score":[111,275]},"I_BS_02":{"L":275,"N":110,"init":[1,110],"score":[111,275]},"I_CA_01":{"L":350,"N":140,"init":[1,140],"score":[141,350]},"I_CA_02":{"L":525,"N":200,"init":[1,200],"score":[201,525]},"I_IL_01":{"L":300,"N":120,"init":[1,120],"score":[121,300]},"I_IL_02":{"L":525,"N":200,"init":[1,200],"score":[201,525]},"I_MB_01":{"L":450,"N":180,"init":[1,180],"score":[181,450]},"I_MB_02":{"L":350,"N":140,"init":[1,140],"score":[141,350]},"I_OC_01":{"L":250,"N":100,"init":[1,100],"score":[101,250]},"I_OC_02":{"L":250,"N":100,"init":[1,100],"score":[101,250]},"I_SI_01":{"L":300,"N":120,"init":[1,120],"score":[121,300]},"I_SI_02":{"L":300,"N":120,"init":[1,120],"score":[121,300]},"O_CL_01":{"L":225,"N":90,"init":[1,90],"score":[91,225]},"O_CL_02":{"L":425,"N":170,"init":[1,170],"score":[171,425]},"O_RA_01":{"L":1400,"N":200,"init":[1,200],"score":[201,1400]},"O_RA_02":{"L":375,"N":150,"init":[1,150],"score":[151,375]},"O_SN_01":{"L":500,"N":200,"init":[1,200],"score":[201,500]},"O_SN_02":{"L":850,"N":200,"init":[1,200],"score":[201,850]},"O_SU_01":{"L":250,"N":100,"init":[1,100],"score":[101,250]},"O_SU_02":{"L":400,"N":160,"init":[1,160],"score":[161,400]}}}')
if os.environ.get('ZBS_MANIFEST'):
    MANIFEST = json.load(open(os.environ['ZBS_MANIFEST']))
json.dump(MANIFEST, open(os.path.join(WORK, 'zbs_manifest.json'), 'w'))
log(f"Manifest: {sum(len(v) for v in MANIFEST.values())} sequences " + str({k: len(v) for k, v in MANIFEST.items()}))
log('Cell 5 OK')

In [ ]:
# ===== Cell 6: find the sequence folders in the attached data and check that every needed frame exists =====
sys.path.insert(0, HERE)
import zbs_detect as Z
importlib.reload(Z)
DIRS = Z.find_sequences(INPUT_ROOT, MANIFEST)
for (ds, s), d in sorted(DIRS.items()):
    print(f'  {ds:10s} {s:20s} {d}')
problems, TOTAL = Z.verify(MANIFEST, DIRS)
if problems:
    for p in problems:
        log('PROBLEM: ' + p)
    raise RuntimeError('Some input data is missing (see PROBLEM lines). Check the dataset attached with Add Input.')
log(f'All {len(DIRS)} sequences found; {TOTAL} frames to process')
log('Cell 6 OK')

In [ ]:
# ===== Cell 7: build the ZBS detector, check the weights, and run a visual sanity check =====
import cv2, numpy as np
PRED, CFG, CLASSES = Z.build_predictor(ZBS, WEIGHTS, DEVICE)
os.chdir(HERE)
if not LOCAL_TEST:
    frac, n_model, n_ck = Z.check_weights(PRED.model, WEIGHTS)
    log(f'Weights: {frac * 100:.1f}% of {n_model} model tensors found in the checkpoint ({n_ck} tensors)')
    if frac < 0.98:
        raise RuntimeError('The checkpoint does not match the model')
tiles, best = [], 0.0
for ds in Z.ORDER:
    seqs = [s for s in MANIFEST.get(ds, {}) if (ds, s) in DIRS]
    if not seqs:
        continue
    s = seqs[0]; plan = Z.frame_plan(ds, MANIFEST[ds][s]); n = [k for k, r in plan if r == 'score'][0]
    img = cv2.imread(Z.index_frames(DIRS[(ds, s)])[n])
    inst = Z.detect(PRED, img)
    keep = (inst.scores >= 0.6).numpy()
    top = sorted(zip(inst.scores.numpy()[keep], inst.pred_classes.numpy()[keep]), reverse=True)[:6]
    log(f'  {ds}/{s} frame {n}: {int(keep.sum())} detections >= 0.6: ' + ', '.join(f'{CLASSES[c]} {v:.2f}' for v, c in top))
    best = max(best, float(inst.scores.max()) if len(inst) else 0.0)
    vis = img.copy()
    for (x1, y1, x2, y2), c in zip(inst.pred_boxes.tensor.numpy()[keep].astype(int), inst.pred_classes.numpy()[keep]):
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 1)
        cv2.putText(vis, CLASSES[c][:12], (x1, max(8, y1 - 2)), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 0), 1)
    tiles.append(cv2.resize(vis, (320, 240)))
cv2.imwrite(os.path.join(WORK, 'zbs_sanity.png'), np.hstack(tiles))
if not LOCAL_TEST and best < 0.6:
    raise RuntimeError('The detector found nothing with score >= 0.6 on the test frames; something is wrong')
log('Sanity image saved as zbs_sanity.png (boxes = detections with score >= 0.6)')
log('Cell 7 OK')

In [ ]:
# ===== Cell 8: speed test and time estimate =====
ds0 = [d for d in Z.ORDER if MANIFEST.get(d)][0]; s0 = list(MANIFEST[ds0])[0]
idx0 = Z.index_frames(DIRS[(ds0, s0)]); plan0 = Z.frame_plan(ds0, MANIFEST[ds0][s0])[:3 if LOCAL_TEST else 25]
imgs = [cv2.imread(idx0[n]) for n, _ in plan0]
Z.detect(PRED, imgs[0])                                         # warm-up
if DEVICE == 'cuda': torch.cuda.synchronize()
t = time.time()
for im in imgs[1:]:
    Z.encode(Z.detect(PRED, im))
if DEVICE == 'cuda': torch.cuda.synchronize()
FPS = (len(imgs) - 1) / (time.time() - t)
est_h = TOTAL / FPS / 3600
log(f'Speed: {FPS:.2f} frames/s -> about {est_h:.1f} h for {TOTAL} frames (time budget {TIME_BUDGET_H} h)')
if est_h > TIME_BUDGET_H - (time.time() - T0) / 3600:
    log('WARNING: the estimate exceeds the time budget; the run will stop early but still save every finished sequence')
log('Cell 8 OK  ->  if everything above is fine: Save Version > Save & Run All (Commit)')

In [ ]:
# ===== Cell 9: main run (LASIESTA first, then LIMU, then WallFlower) =====
DEADLINE = T0 + TIME_BUDGET_H * 3600
META = dict(config=Z.CONFIG, cache_conf=Z.CACHE_CONF, torch=torch.__version__, detectron2=detectron2.__version__,
            zbs_commit=ZBS_COMMIT, gpu=torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu')
STATUS = {}
for ds in Z.ORDER:
    for s, spec in MANIFEST.get(ds, {}).items():
        key = f'{ds}/{s}'
        try:
            STATUS[key] = Z.run_sequence(PRED, ds, s, DIRS[(ds, s)], spec, OUT, META, DEADLINE, log=log, max_frames=MAX_FRAMES)
        except Exception as e:
            STATUS[key] = f'error: {e!r}'
            log(f'ERROR in {key}: {e!r}'); traceback.print_exc()
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        log(f'{key}: {STATUS[key]}   (elapsed {(time.time() - T0) / 3600:.2f} h)')
log('Cell 9 OK')

In [ ]:
# ===== Cell 10: package the results =====
import zipfile
done = sorted(glob.glob(os.path.join(OUT, '*.pkl.gz')))
expected = sum(len(v) for v in MANIFEST.values())
summary = dict(sequences_done=len(done), sequences_expected=expected, status=STATUS, hours=round((time.time() - T0) / 3600, 2),
               fps=round(FPS, 3), meta=META, files=[os.path.basename(p) for p in done])
json.dump(summary, open(os.path.join(WORK, 'zbs_summary.json'), 'w'), indent=1)
zpath = shutil.make_archive(os.path.join(WORK, 'zbs_dets'), 'zip', OUT)
with zipfile.ZipFile(zpath) as zf:
    bad = zf.testzip(); n_in = len(zf.namelist())
if bad is None and n_in == len(done):
    shutil.rmtree(OUT)                       # the zip is complete and verified; keep only the zip in the output
log(f'FINISHED: {len(done)}/{expected} sequences, zip {os.path.getsize(zpath) / 1e6:.0f} MB ({n_in} files, check '
    f'{"OK" if bad is None else "FAILED"}). Download zbs_dets.zip, zbs_summary.json and zbs_log.txt from the Output tab.')